<a href="https://colab.research.google.com/github/Walid75364/GenIA/blob/Bootcamp_Weeks/W7_D2_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Daily Challenge**

In [ ]:
# 📦 Installation des bibliothèques nécessaires
%pip install peft==0.4.0
%pip install datasets transformers accelerate


  Using cached peft-0.4.0-py3-none-any.whl.metadata (21 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-manylinux2

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [1]:
# 🔄 Met à jour les bibliothèques concernées
%pip install -U datasets huggingface_hub fsspec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.7.0
    Uninstalling fsspec-2025.7.0:
      Successfully uninstalled fsspec-2025.7.0
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.7.0 requires fsspec==2025.7.0, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
# 📁 Création du dossier de cache pour stocker les résultats
import os
os.makedirs("cache", exist_ok=True)

# 🔍 Chargement du modèle pré-entraîné et du tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# 🗂️ Chargement du dataset et prétraitement
from datasets import load_dataset
#data = load_dataset("Abirate/english_quotes", split="train[:10%]")
full_data = load_dataset("Abirate/english_quotes", split="train")
data = full_data.select(range(int(0.005 * len(full_data))) )  # 0.5 % manuel


# 🧹 Tokenisation des échantillons
tokenized_data = data.map(lambda samples: tokenizer(samples["quote"], truncation=True, padding="max_length"), batched=True)

# 🧪 Affichage d'un échantillon d'entraînement
train_sample = tokenized_data.select(range(5))
print(train_sample)

# 🔧 Configuration LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],  # cible typique pour BLOOMZ
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 🛠️ Application de LoRA au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

# ⚙️ Configuration des paramètres d'entraînement
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

output_directory = os.path.join("cache", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,  # Taux d'apprentissage plus élevé
    num_train_epochs=3,
    use_cpu=True
)

# 🎓 Initialisation du Trainer pour lancer l'entraînement
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()

# 💾 Sauvegarde du modèle fine-tuné
import time
time_now = time.strftime("%Y-%m-%d-%H-%M-%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# 🔮 Chargement du modèle pour inférence
from peft import PeftModel

inference_model = PeftModel.from_pretrained(foundation_model, peft_model_path, is_trainable=False)

# 🧠 Génération de texte
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = inference_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

# 📝 Affichage du texte généré
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.01757585078102687


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss


["Two things are infinite:  that you can do and live the life you've had to live and do.”  That’s about what he said in his first speech on September 27: “I’ve been telling people my first two words in a long line, but that’s fine.”"]


**🤖 Interprétation de l’output généré**

"Two things are infinite: happiness, and romance. And the world doesn't deserve to be like that...”

Ce que ça montre :
Le modèle a bien appris à prolonger une citation dans le style du dataset.

Il tente d’évoquer des thèmes émotionnels et philosophiques cohérents avec les citations du corpus.

Le reste est un peu chaotique ("if it's enough to hurt him.”””””…”), ce qui indique un manque de régularité, probablement dû au faible volume d’entraînement ou à un nettoyage insuffisant des guillemets et ponctuations dans le dataset. En effet, après plusieurs tentatives de train à 10%, 5%, 3% et 2%, j'ai dû choisir un train à 0.5% faute de gpu malgré leur activation.
Néanmoins j'ai réussi à avoir un résultat avec un train à 1% comme ci-dessous.
Limit du code avec gpu : 1%.

In [1]:
# 📁 Création du dossier de cache pour stocker les résultats
import os
os.makedirs("cache", exist_ok=True)

# 🔍 Chargement du modèle pré-entraîné et du tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# 🗂️ Chargement du dataset et prétraitement
from datasets import load_dataset
data = load_dataset("Abirate/english_quotes", split="train[:1%]")

# 🧹 Tokenisation des échantillons
tokenized_data = data.map(lambda samples: tokenizer(samples["quote"], truncation=True, padding="max_length"), batched=True)

# 🧪 Affichage d'un échantillon d'entraînement
train_sample = tokenized_data.select(range(5))
print(train_sample)

# 🔧 Configuration LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],  # cible typique pour BLOOMZ
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 🛠️ Application de LoRA au modèle de base
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

# ⚙️ Configuration des paramètres d'entraînement
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

output_directory = os.path.join("cache", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,  # Taux d'apprentissage plus élevé
    num_train_epochs=3,
    use_cpu=True
)

# 🎓 Initialisation du Trainer pour lancer l'entraînement
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()

# 💾 Sauvegarde du modèle fine-tuné
import time
time_now = time.strftime("%Y-%m-%d-%H-%M-%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# 🔮 Chargement du modèle pour inférence
from peft import PeftModel

inference_model = PeftModel.from_pretrained(foundation_model, peft_model_path, is_trainable=False)

# 🧠 Génération de texte
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = inference_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

# 📝 Affichage du texte généré
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})
trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.01757585078102687


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss


['Two things are infinite:  and are things that are just as you like it.  I am . ” . ” .  and like everything that you can do that one thing like it.” is a matter you live in love.” is the universe.” can be something']
